# SKGB Neo4j GraphRAG Retrieval Test

Test the notebook-oriented retrieval package in `skgb/retrieval` against the existing Neo4j graph and inspect the evidence returned for each answer.

This notebook assumes:
- the repository root `.env` is the canonical configuration source
- Neo4j is reachable
- retrieval defaults to the flexible `entity_graph` strategy unless vector mode is explicitly configured
- the graph contains searchable node properties and relationships that can be used as evidence


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'skgb').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Could not locate the repository root.')
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ENV_FILE = REPO_ROOT / '.env'
print('Repo root:', REPO_ROOT)
print('Env file:', ENV_FILE)


Repo root: /workspace
Env file: /workspace/.env


In [2]:
import pandas as pd

from skgb.retrieval import ask_graph, build_rag, load_retrieval_config, search_context

cfg = load_retrieval_config(ENV_FILE)
cfg_view = {
    'env_file': str(cfg.env_file),
    'neo4j_uri': cfg.neo4j_uri,
    'neo4j_database': cfg.neo4j_database,
    'neo4j_auth_disabled': cfg.neo4j_auth_disabled,
    'retrieval_strategy': cfg.retrieval_strategy,
    'neo4j_vector_index': cfg.neo4j_vector_index,
    'neo4j_fulltext_index': cfg.neo4j_fulltext_index,
    'ollama_host': cfg.ollama_host,
    'ollama_llm_model': cfg.ollama_llm_model,
    'ollama_embeddings_model': cfg.ollama_embeddings_model,
    'retriever_top_k': cfg.retriever_top_k,
    'response_fallback': cfg.response_fallback,
}
pd.Series(cfg_view)


env_file                                                     /workspace/.env
neo4j_uri                                                 neo4j://neo4j:7687
neo4j_database                                                         neo4j
neo4j_auth_disabled                                                     True
retrieval_strategy                                              entity_graph
neo4j_vector_index                                           chunkEmbeddings
neo4j_fulltext_index                                                    None
ollama_host                                               https://ollama.com
ollama_llm_model                                            gemma4:31b-cloud
ollama_embeddings_model                       nomic-embed-text-v2-moe:latest
retriever_top_k                                                            5
response_fallback          I cannot answer this question because no relev...
dtype: object

In [3]:
runtime = build_rag(env_file=ENV_FILE, validate=True)
print('Retrieval runtime ready.')


Retrieval runtime ready.


In [4]:
def evidence_table(items):
    rows = []
    for rank, item in enumerate(items, start=1):
        metadata = dict(item.metadata or {})
        rows.append({
            'rank': rank,
            'score': metadata.get('score'),
            'source': metadata.get('source'),
            'entities': ', '.join(metadata.get('entities') or []),
            'content': item.content,
        })
    return pd.DataFrame(rows)


def ask_with_evidence(question: str, top_k: int | None = None):
    context_result = search_context(question, top_k=top_k, runtime=runtime)
    answer_result = ask_graph(question, top_k=top_k, runtime=runtime, return_context=True)
    return answer_result, evidence_table(context_result.items)


In [5]:
QUESTION = 'What are the main findings in this knowledge graph?'
TOP_K = 5
QUESTION


'What are the main findings in this knowledge graph?'

In [6]:
answer_result, evidence_df = ask_with_evidence(QUESTION, top_k=TOP_K)
evidence_df


,rank,score,source,entities,content
0,1,4.3579,Publication,"Muthulingam: s., Wieland: a., Mukherjee: u.",Seed node: Publication: does organizational fo...
1,2,0.6328,Publication,AdaptiveRobotAdoptionPolicy: adaptive robot ad...,Seed node: Publication: 'two perspectives on s...
2,3,0.5836,ScrIndicator,"Study: spillover effects in the supply chain, ...",Seed node: ScrIndicator: supply chain resilien...
3,4,0.5659,Publication,"Journal: journal of business logistics, Zhou: ...",Seed node: Publication: 'driving total factor ...
4,5,0.5562,Study,ScrIndicator: supply chain resilience (scr) in...,Seed node: Study: spillover effects in the sup...


In [7]:
print(answer_result.answer)


Based on the provided evidence, the main findings center on the relationship between robot adoption and supply chain resilience, specifically within the context of Chinese manufacturing.

The primary findings are:
* **Robot Adoption and Resilience:** Robot adoption is positively associated with the supply chain resilience (SCR) indicator. Furthermore, "RobotAdoption: robot" is noted to promote, improve, and have a positive impact on the publication "two perspectives on supply chain resilience."
* **Spillover Effects:** The study "spillover effects in the supply chain" specifically finds that there are positive spillover effects of upstream enterprise robot adoption on downstream supply chain resilience.
* **Policy and Moderation:** An "adaptive robot adoption policy" is linked to promoting and improving the publication "two perspectives on supply chain resilience." Additionally, the SCR indicator considers the moderating role of "firms' digital transformation."
* **Study Parameters:** 

In [8]:
for index, item in enumerate(answer_result.context or [], start=1):
    print(f'--- Evidence {index} ---')
    print(item.content)
    print()


--- Evidence 1 ---
Seed node: Publication: does organizational forgetting affect quality knowledge gained through spillover? evidence from the automotive industry
Properties:
- category: publication
- id: does_organizational_forgetting_affect_quality_knowledge_gained_through_spillover_evidence_from_the_automotive_industry
Connected graph facts:
- Muthulingam: s. --[AUTHORED]--> Publication: does organizational forgetting affect quality knowledge gained through spillover? evidence from the automotive industry (edge_row=193, original_relation=authored)
- Wieland: a. --[AUTHORED]--> Publication: does organizational forgetting affect quality knowledge gained through spillover? evidence from the automotive industry (edge_row=195, original_relation=authored)
- Mukherjee: u. --[AUTHORED]--> Publication: does organizational forgetting affect quality knowledge gained through spillover? evidence from the automotive industry (edge_row=196, original_relation=authored)

--- Evidence 2 ---
Seed node

In [9]:
# runtime.close()
